# SQL vs. Cypher — Side by Side

Notebook 09 taught Cypher in isolation. This one runs the **same six questions** against MovieLens twice — once as SQL on an in-memory SQLite mirror, once as Cypher on the live Neo4j graph — so the contrast is something a reader can *see*, not just be told.

See [`docs/10_sql_vs_cypher.md`](../docs/10_sql_vs_cypher.md) for the scope decisions behind this notebook (why SQLite, why these six examples, what's deliberately out).

---

## How to read each example

Every example block has three cells:

1. **Markdown** — the question in English, plus a one-line note on what to compare.
2. **SQL on SQLite** — the tabular answer.
3. **Cypher on Neo4j** — the graph answer.

Both sides print the same answer set (sorted, so the orders match). The lesson is in the *shape* of each query, not the output.

## Recurring anchors

Same four anchors as notebook 09 — kept identical on purpose so a reader can compare *queries*, not data:

- `Toy Story` — `movieId = 1`
- `Tom Hanks` — actor
- `user 196` — the most active rater in MovieLens 100K
- `Comedy` — genre

## Setup

Two backends to wire up:

- **Neo4j** — same `run_query` helper as notebooks 04–09, reads credentials from `.env`.
- **SQLite** — an in-memory mirror of the same data. Nine tables, built from the same source files the graph load used (`u.data`, `u.user` from MovieLens, plus the TMDB-enriched `enriched_movies.csv`).

The SQLite side does **not** import from `src/etl.py`. The list-exploding work in the next cell *is part of the lesson* — bridge tables are exactly the tabular tax the graph side skips.

In [16]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from graph import run_query, close_driver

# Smoke-test the Neo4j side: should print the live node-label set.
labels = run_query("CALL db.labels() YIELD label RETURN label ORDER BY label")
print("Neo4j labels:", [r["label"] for r in labels])

Neo4j labels: ['Actor', 'Director', 'Gender', 'Genre', 'Movie', 'Occupation', 'User']


In [17]:
import sqlite3
import ast
import re
import pandas as pd

DATA_DIR = Path.cwd().parent / "data"
RAW_DIR = DATA_DIR / "raw" / "ml-100k"
ENRICHED_DIR = DATA_DIR / "enriched"

con = sqlite3.connect(":memory:")

# --- users -------------------------------------------------------------------
users = pd.read_csv(
    RAW_DIR / "u.user",
    sep="|",
    names=["userId", "age", "gender", "occupation", "zip"],
    encoding="latin-1",
)[["userId", "age", "gender", "occupation"]]
users.to_sql("users", con, index=False)

# --- ratings -----------------------------------------------------------------
ratings = pd.read_csv(
    RAW_DIR / "u.data",
    sep="\t",
    names=["userId", "movieId", "rating", "timestamp"],
)
ratings.to_sql("ratings", con, index=False)

# --- movies (title + year split from raw_title) ------------------------------
enriched = pd.read_csv(ENRICHED_DIR / "enriched_movies.csv")
_TITLE_YEAR_RE = re.compile(r"^(.*) \((\d{4})\)$")

def _split_title(raw):
    m = _TITLE_YEAR_RE.match(str(raw))
    return (m.group(1), int(m.group(2))) if m else (str(raw), None)

titles_years = enriched["raw_title"].apply(_split_title)
movies = pd.DataFrame({
    "movieId": enriched["movieId"],
    "title":   [t for t, _ in titles_years],
    "year":    [y for _, y in titles_years],
    "tmdb_id": enriched["tmdb_id"],
})
movies.to_sql("movies", con, index=False)

# --- helper: explode a stringified-list column into a long bridge table ------
# Mirrors src/etl.py conventions (same parse, same skip-NaN behavior).
def _explode(col_name, value_col, with_billing_order=False):
    rows = []
    for _, row in enriched[["movieId", col_name]].iterrows():
        cell = row[col_name]
        if pd.isna(cell):
            continue
        for idx, name in enumerate(ast.literal_eval(cell), start=1):
            entry = {"movieId": int(row["movieId"]), value_col: name}
            if with_billing_order:
                entry["billing_order"] = idx
            rows.append(entry)
    return pd.DataFrame(rows)

# --- genres + movie_genres ---------------------------------------------------
movie_genres = _explode("genre_names", "genre_name")
movie_genres.to_sql("movie_genres", con, index=False)
pd.DataFrame({"name": sorted(movie_genres["genre_name"].unique())}).to_sql(
    "genres", con, index=False
)

# --- actors + movie_actors (carries billing_order, like FEATURES_ACTOR) ------
movie_actors = _explode("cast", "actor_name", with_billing_order=True)
movie_actors.to_sql("movie_actors", con, index=False)
pd.DataFrame({"name": sorted(movie_actors["actor_name"].unique())}).to_sql(
    "actors", con, index=False
)

# --- directors + movie_directors ---------------------------------------------
movie_directors = _explode("directors", "director_name")
movie_directors.to_sql("movie_directors", con, index=False)
pd.DataFrame({"name": sorted(movie_directors["director_name"].unique())}).to_sql(
    "directors", con, index=False
)

# --- soft-verify: row counts per table ---------------------------------------
print(f"{'Table':<18} {'Rows':>10}")
print("-" * 30)
for table in ["users", "ratings", "movies",
              "genres", "movie_genres",
              "actors", "movie_actors",
              "directors", "movie_directors"]:
    n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<18} {n:>10,}")

Table                    Rows
------------------------------
users                     943
ratings               100,000
movies                  1,682
genres                     19
movie_genres            4,096
actors                  9,301
movie_actors           16,243
directors               1,128
movie_directors         1,790


## Example 1 — Movies in genre "Comedy"

The trivial case. One bridge table on the SQL side, one edge hop on the Cypher side — both queries are essentially the same shape. This is the baseline against which the next five examples open the gap.

**Compare:** line count and conceptual weight. They should feel about even.

In [18]:
sql_1 = """
SELECT m.movieId, m.title
FROM movies m
JOIN movie_genres mg ON mg.movieId = m.movieId
WHERE mg.genre_name = 'Comedy'
ORDER BY m.title
LIMIT 10
"""
pd.read_sql(sql_1, con)

,movieId,title
0,1300,'Til There Was You
1,225,101 Dalmatians
2,1011,2 Days in the Valley
3,314,3 Ninjas: High Noon At Mega Mountain
4,1664,8 Heads in a Duffel Bag
5,1251,A Chef in Love
6,67,Ace Ventura: Pet Detective
7,364,Ace Ventura: When Nature Calls
8,386,Addams Family Values
9,535,Addicted to Love


In [19]:
cypher_1 = """
MATCH (m:Movie)-[:IN_GENRE]->(:Genre {name: 'Comedy'})
RETURN m.movieId AS movieId, m.title AS title
ORDER BY m.title
LIMIT 10
"""
pd.DataFrame(run_query(cypher_1))

,movieId,title
0,225,101 Dalmatians
1,1664,8 Heads in a Duffel Bag
2,1251,A Chef in Love
3,67,Ace Ventura: Pet Detective
4,364,Ace Ventura: When Nature Calls
5,386,Addams Family Values
6,535,Addicted to Love
7,382,"Adventures of Priscilla, Queen of the Desert, The"
8,261,Air Bud
9,801,"Air Up There, The"


## Example 2 — Other movies featuring Toy Story's actors

Two hops on the graph side: Toy Story → shared actor → other movie. On the SQL side, the same logic takes **three JOINs** (`movies` ↔ `movie_actors` ↔ `movie_actors` self-join ↔ `movies`) and a `<>` filter to keep Toy Story itself out of the result.

**Compare:** the SQL needs to spell out that the same actor appears in two different movies (the `ma2.actor_name = ma1.actor_name AND ma2.movieId <> anchor.movieId` predicate). Cypher binds `(a:Actor)` once and lets the pattern itself encode the "shared" relationship.

This is exactly the query that Notebook 09 §5 Shape 2 runs against the graph — same answer set, two languages.

In [20]:
sql_2 = """
SELECT DISTINCT
    other.movieId,
    other.title,
    ma1.actor_name AS shared_actor
FROM movies anchor
JOIN movie_actors ma1 ON ma1.movieId   = anchor.movieId
JOIN movie_actors ma2 ON ma2.actor_name = ma1.actor_name
                     AND ma2.movieId   <> anchor.movieId
JOIN movies       other ON other.movieId = ma2.movieId
WHERE anchor.movieId = 1
ORDER BY other.year, other.title, shared_actor
LIMIT 10
"""
pd.read_sql(sql_2, con)

,movieId,title,shared_actor
0,173,"Princess Bride, The",Wallace Shawn
1,1030,"Beverly Hillbillies, The",Jim Varney
2,735,Philadelphia,Tom Hanks
3,88,Sleepless in Seattle,Tom Hanks
4,1248,Blink,Laurie Metcalf
5,69,Forrest Gump,Tom Hanks
6,1113,Mrs. Parker and the Vicious Circle,Wallace Shawn
7,63,"Santa Clause, The",Tim Allen
8,464,Vanya on 42nd Street,Wallace Shawn
9,28,Apollo 13,Tom Hanks


In [21]:
cypher_2 = """
MATCH (m:Movie {movieId: 1})-[:FEATURES_ACTOR]->(a:Actor)<-[:FEATURES_ACTOR]-(other:Movie)
WHERE other <> m
RETURN other.movieId AS movieId,
       other.title   AS title,
       a.name        AS shared_actor
ORDER BY other.year, other.title, a.name
LIMIT 10
"""
pd.DataFrame(run_query(cypher_2))

,movieId,title,shared_actor
0,173,"Princess Bride, The",Wallace Shawn
1,1030,"Beverly Hillbillies, The",Jim Varney
2,735,Philadelphia,Tom Hanks
3,88,Sleepless in Seattle,Tom Hanks
4,1248,Blink,Laurie Metcalf
5,69,Forrest Gump,Tom Hanks
6,1113,Mrs. Parker and the Vicious Circle,Wallace Shawn
7,63,"Santa Clause, The",Tim Allen
8,464,Vanya on 42nd Street,Wallace Shawn
9,28,Apollo 13,Tom Hanks


## Example 3 — Users who rated the same movies as user 196

"Collaborative-filter" style — find user 196's nearest neighbors in rating overlap. SQL needs a `ratings` self-join with a `<>` predicate to keep user 196 out of the result set; Cypher binds the two users as separate variables but gets the `<> u1` for free from **relationship uniqueness** (the same lesson from Notebook 09 §5 Shape 1 — both `:RATED` slots can't bind to the same edge).

**Compare:** the SQL `WHERE r2.userId <> r1.userId` is load-bearing here; without it user 196 dominates every result with their own ratings. The equivalent Cypher filter is *redundant* in this two-hop shape.

In [22]:
sql_3 = """
SELECT r2.userId AS other_user,
       COUNT(*) AS shared_ratings
FROM ratings r1
JOIN ratings r2 ON r2.movieId = r1.movieId
                AND r2.userId <> r1.userId
WHERE r1.userId = 196
GROUP BY r2.userId
ORDER BY shared_ratings DESC, other_user
LIMIT 10
"""
pd.read_sql(sql_3, con)

,other_user,shared_ratings
0,655,31
1,13,30
2,303,28
3,378,28
4,416,26
5,18,25
6,327,25
7,354,25
8,450,25
9,279,24


In [23]:
cypher_3 = """
MATCH (u1:User {userId: 196})-[:RATED]->(m:Movie)<-[:RATED]-(u2:User)
RETURN u2.userId AS other_user,
       count(m)  AS shared_ratings
ORDER BY shared_ratings DESC, other_user
LIMIT 10
"""
pd.DataFrame(run_query(cypher_3))

,other_user,shared_ratings
0,655,31
1,13,30
2,303,28
3,378,28
4,416,26
5,18,25
6,327,25
7,354,25
8,450,25
9,279,24


## Example 4 — Movies in genres user 196 has rated (that they haven't seen yet)

The candidate-generation step in a genre-affinity recommender: find genres user 196 has historically watched, then list movies in those genres that user 196 hasn't yet rated.

SQL needs **four JOINs** (ratings → genre-bridge → genre-bridge self-join → movies) plus a `NOT IN` subquery against `ratings` to exclude already-seen films. Cypher walks the three-hop pattern `(user)→(rated movie)→(genre)←(candidate)` and uses a `NOT (u)-[:RATED]->(other)` pattern predicate for the exclusion.

**Compare:** at three hops, the SQL is becoming hard to read at a glance. Cypher reads almost like the English description.

In [24]:
sql_4 = """
SELECT DISTINCT
    m2.movieId,
    m2.title,
    mg2.genre_name AS via_genre
FROM ratings r
JOIN movie_genres mg1 ON mg1.movieId   = r.movieId
JOIN movie_genres mg2 ON mg2.genre_name = mg1.genre_name
JOIN movies       m2  ON m2.movieId    = mg2.movieId
WHERE r.userId = 196
  AND m2.movieId NOT IN (SELECT movieId FROM ratings WHERE userId = 196)
ORDER BY m2.title, via_genre
LIMIT 10
"""
pd.read_sql(sql_4, con)

,movieId,title,via_genre
0,1300,'Til There Was You,Comedy
1,1300,'Til There Was You,Romance
2,1353,1-900,Drama
3,1353,1-900,Romance
4,225,101 Dalmatians,Comedy
5,225,101 Dalmatians,Family
6,178,12 Angry Men,Drama
7,330,187,Drama
8,1011,2 Days in the Valley,Comedy
9,1011,2 Days in the Valley,Crime


In [25]:
cypher_4 = """
MATCH (u:User {userId: 196})-[:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(other:Movie)
WHERE NOT (u)-[:RATED]->(other)
RETURN DISTINCT
    other.movieId AS movieId,
    other.title   AS title,
    g.name        AS via_genre
ORDER BY title, via_genre
LIMIT 10
"""
pd.DataFrame(run_query(cypher_4))

,movieId,title,via_genre
0,1300,'Til There Was You,Drama
1,1300,'Til There Was You,Romance
2,1353,1-900,Romance
3,225,101 Dalmatians,Children's
4,225,101 Dalmatians,Comedy
5,178,12 Angry Men,Drama
6,330,187,Drama
7,141,"20,000 Leagues Under the Sea",Adventure
8,141,"20,000 Leagues Under the Sea",Children's
9,141,"20,000 Leagues Under the Sea",Fantasy


## Example 5 — Explainable recommendations for user 196

CLAUDE.md requires **every recommendation to return an explainability path** — e.g. *"You rated Toy Story 5★ → Tom Hanks → Apollo 13."* This example builds that string both ways.

SQL has to compose the string with `||` concatenation across the JOINed rows — every piece of the explanation comes from a different table, so the column list grows long fast.

Cypher gives you a choice: either build the same string with `+`, or return the path itself as a first-class object (`RETURN path`) and let the application render it. We use the string form here for one-to-one SQL parity, but the path-object form is what makes the explainability requirement easy to satisfy downstream.

**Compare:** look at the column list in each query. The SQL `||` chain references four different aliases (`m1`, `r`, `ma1`, `m2`); the Cypher `+` chain references three variables bound in a single pattern.

In [26]:
sql_5 = """
SELECT
    'You rated ' || m1.title || ' (' || r.rating || '*) -> '
                 || ma1.actor_name || ' -> ' || m2.title AS explanation
FROM ratings r
JOIN movies       m1  ON m1.movieId    = r.movieId
JOIN movie_actors ma1 ON ma1.movieId    = m1.movieId
JOIN movie_actors ma2 ON ma2.actor_name = ma1.actor_name
                     AND ma2.movieId   <> m1.movieId
JOIN movies       m2  ON m2.movieId    = ma2.movieId
WHERE r.userId = 196
  AND m2.movieId NOT IN (SELECT movieId FROM ratings WHERE userId = 196)
ORDER BY r.rating DESC, m2.title
LIMIT 10
"""
pd.read_sql(sql_5, con)

,explanation
0,You rated Being There (5*) -> Jack Warden -> 1...
1,You rated Ace Ventura: Pet Detective (5*) -> J...
2,You rated Ace Ventura: Pet Detective (5*) -> U...
3,"You rated American President, The (5*) -> Davi..."
4,"You rated English Patient, The (5*) -> Kristin..."
5,"You rated English Patient, The (5*) -> Kristin..."
6,"You rated American President, The (5*) -> Rich..."
7,You rated Being There (5*) -> Shirley MacLaine...
8,"You rated American President, The (5*) -> Mart..."
9,You rated Being There (5*) -> Shirley MacLaine...


In [27]:
cypher_5 = """
MATCH (u:User {userId: 196})-[r:RATED]->(m1:Movie)
      -[:FEATURES_ACTOR]->(a:Actor)<-[:FEATURES_ACTOR]-(m2:Movie)
WHERE NOT (u)-[:RATED]->(m2)
RETURN 'You rated ' + m1.title + ' (' + toString(r.rating) + '*) -> '
                    + a.name + ' -> ' + m2.title AS explanation
ORDER BY r.rating DESC, m2.title
LIMIT 10
"""
pd.DataFrame(run_query(cypher_5))

,explanation
0,You rated Being There (5.0*) -> Jack Warden ->...
1,You rated Ace Ventura: Pet Detective (5.0*) ->...
2,You rated Ace Ventura: Pet Detective (5.0*) ->...
3,"You rated American President, The (5.0*) -> Da..."
4,"You rated English Patient, The (5.0*) -> Krist..."
5,"You rated English Patient, The (5.0*) -> Krist..."
6,"You rated American President, The (5.0*) -> Ri..."
7,You rated Being There (5.0*) -> Shirley MacLai...
8,"You rated American President, The (5.0*) -> Ma..."
9,You rated Being There (5.0*) -> Shirley MacLai...


## Example 6 — The triangle: three actors who have all collaborated

The canonical "easy in graph, brutal in SQL" example, and the payoff for this whole notebook. Three actors A, B, C where every pair has appeared in at least one movie together.

The graph version (this is the same query as Notebook 07 spot-check 3) is one pattern: traverse the triangle. The SQL version needs to **pre-compute the "collaborated with" relation as a CTE**, then three self-joins on top to express the triangle. The CTE is itself a self-join on `movie_actors` — that's the cost graph users pay once (we did it in Notebook 07's `:COLLABORATED_WITH` materialization) and tabular users pay every time.

**Compare:** the `a.name < b.name AND b.name < c.name` deduplication is in *both* queries (6 permutations per triangle, only one canonical) — that's the only thing they have in common. The rest is `MATCH (a)-(b)-(c)-(a)` vs. three JOINs across the CTE plus three equality chains.

In [28]:
sql_6 = """
WITH collab AS (
    -- Computes the same logical relation as :COLLABORATED_WITH from Notebook 07.
    -- One row per canonical (a < b) pair of actors who share at least one movie.
    SELECT DISTINCT
        ma1.actor_name AS a,
        ma2.actor_name AS b
    FROM movie_actors ma1
    JOIN movie_actors ma2 ON ma2.movieId   = ma1.movieId
                          AND ma1.actor_name < ma2.actor_name
)
SELECT
    ab.a AS actor_a,
    ab.b AS actor_b,
    bc.b AS actor_c
FROM collab ab
JOIN collab bc ON bc.a = ab.b
JOIN collab ac ON ac.a = ab.a
              AND ac.b = bc.b
ORDER BY actor_a, actor_b, actor_c
LIMIT 10
"""
pd.read_sql(sql_6, con)

,actor_a,actor_b,actor_c
0,A. Bailey Chapin,Anthony McKay,Barnaby Rudge
1,A. Bailey Chapin,Anthony McKay,Bo White
2,A. Bailey Chapin,Anthony McKay,Curt Gareth
3,A. Bailey Chapin,Anthony McKay,Jay Pierce
4,A. Bailey Chapin,Anthony McKay,Marilyn Meyers
5,A. Bailey Chapin,Anthony McKay,Michael Kell
6,A. Bailey Chapin,Anthony McKay,Robert McLane
7,A. Bailey Chapin,Anthony McKay,Scott Eisman
8,A. Bailey Chapin,Barnaby Rudge,Bo White
9,A. Bailey Chapin,Barnaby Rudge,Curt Gareth


In [29]:
cypher_6 = """
MATCH (a:Actor)-[:COLLABORATED_WITH]-(b:Actor)-[:COLLABORATED_WITH]-(c:Actor)-[:COLLABORATED_WITH]-(a:Actor)
WHERE a.name < b.name AND b.name < c.name
RETURN a.name AS actor_a, b.name AS actor_b, c.name AS actor_c
ORDER BY actor_a, actor_b, actor_c
LIMIT 10
"""
pd.DataFrame(run_query(cypher_6))

,actor_a,actor_b,actor_c
0,A. Bailey Chapin,Anthony McKay,Barnaby Rudge
1,A. Bailey Chapin,Anthony McKay,Bo White
2,A. Bailey Chapin,Anthony McKay,Curt Gareth
3,A. Bailey Chapin,Anthony McKay,Jay Pierce
4,A. Bailey Chapin,Anthony McKay,Marilyn Meyers
5,A. Bailey Chapin,Anthony McKay,Michael Kell
6,A. Bailey Chapin,Anthony McKay,Robert McLane
7,A. Bailey Chapin,Anthony McKay,Scott Eisman
8,A. Bailey Chapin,Barnaby Rudge,Bo White
9,A. Bailey Chapin,Barnaby Rudge,Curt Gareth


## Wrap-up

A few honest things to say at the end.

- **Example 1 is roughly a tie.** The graph doesn't pay off until questions get multi-hop. If everything you ever ask is "give me all rows where X=Y," tabular is fine.
- **The gap widens, predictably, with hop count.** SQL grows JOINs linearly per hop; Cypher grows pattern characters. By Example 4 the SQL is getting hard to scan; by Example 6 it's a CTE plus three self-joins to express what the graph spells as a triangle.
- **Explainability (Example 5) is where the graph wins on a different axis.** SQL can produce the same string, but the graph hands you the path as a first-class object — which is what the CLAUDE.md "return an explainability path with every recommendation" requirement asks for.
- **What this notebook does not prove.** Not a performance benchmark — in-memory SQLite versus networked AuraDB Free tier is apples-to-oranges. The contrast is *expressiveness* and *fit for the question*, not latency.

The recommender notebooks (T-19 onward) lean on this: every strategy is a multi-hop Cypher pattern returning a path. Now you've seen what that buys.

In [30]:
close_driver()
con.close()